# [PBT] 화장품 이커머스 ㅣ 02. 전제 검증

## 이 노트북이 왜 필요한가

계획서에 적어둔 설계는 **두 가지 전제** 위에 서 있다.
전제가 참인지 확인하지 않고 테이블을 만들면, 만든 뒤에 처음부터 다시 해야 한다.

| 전제 | 확인할 것 | 깨지면 |
|---|---|---|
| **A. `user_id`가 사람 단위 식별자다** | 하나의 `user_id`가 여러 날에 걸쳐 유지되는가 | "7일 내 구매"라는 질문 자체가 성립하지 않는다 |
| **B. `user_session`이 한 번의 방문이다** | 세션이 시간으로 끊긴 단위인가 | 조회 변수 4종을 세션 기준으로 만들 수 없다 |

> 
> 분석 대상 테이블(1행 = 1고객) 생성은 **03번 노트북**에서 한다.
> 이 노트북의 결론이 나와야 그 테이블의 변수 정의가 확정되기 때문이다.

## #01. 준비작업

### 1. 라이브러리 참조

In [1]:
import os, sys

# 프로젝트 폴더 — notebooks/ 안에서 실행하면 상위 폴더를 가리킨다
from pathlib import Path
BASE = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
os.chdir(BASE)             # 폰트 경로가 상대경로라 여기서 실행해야 한다
sys.path.insert(0, BASE)

from pandas import read_parquet, DataFrame, set_option

from helpers import my_qtcheck, my_plot

set_option("display.max_columns", 50)

### 2. 데이터 불러오기

> ⚠️ `category_id`는 parquet에서 다시 읽으면 숫자로 풀린다.
>
> "category 타입으로의 변환은 **데이터를 불러올 때마다** 진행해야 한다."
>
> 참고로, 이 노트북에서는 `category_id`를 쓰지 않는다.

In [2]:

OUT = BASE + r"\output"

df = read_parquet(OUT + r"\01_qtcheck.parquet")
df = my_qtcheck.set_type(df, as_category=["category_id"])

<class 'pandas.DataFrame'>
RangeIndex: 8259159 entries, 0 to 8259158
Data columns (total 9 columns):
 #   Column         Dtype         
---  ------         -----         
 0   event_time     datetime64[us]
 1   event_type     category      
 2   product_id     int32         
 3   category_id    category      
 4   category_code  category      
 5   brand          category      
 6   price          float64       
 7   user_id        int64         
 8   user_session   str           
dtypes: category(4), datetime64[us](1), float64(1), int32(1), int64(1), str(1)
memory usage: 615.3 MB


### 3. 범위 확인

- 01번 노트북에서 보정된 시간(UTC+3)이 제대로 적용되었는지 확인한다.

In [3]:
print("행 수     :", format(len(df), ","))
print("관측 시작 :", df["event_time"].min())
print("관측 종료 :", df["event_time"].max())
print("관측 일수 :", (df["event_time"].max() - df["event_time"].min()).days, "일")

행 수     : 8,259,159
관측 시작 : 2019-10-01 03:00:00
관측 종료 : 2019-12-01 02:59:58
관측 일수 : 60 일


## #02. 전제 A — `user_id`가 사람 단위 식별자인가

### 왜 확인하는가

목표변수의 정의가 이것이다.

> t₀부터 **7일 안에 같은 고객이** 같은 상품을 산 기록이 있으면 Y=1

여기서 "같은 고객"을 `user_id`로 판정한다.
그런데 이 사이트가 **방문할 때마다 새 ID를 발급**하는 방식이라면,
월요일에 담은 사람과 목요일에 그것을 구매한 사람이 **서로 다른 `user_id`로 기록**된다.
그러면 우리는 그 구매를 영영 찾지 못하고, Y=1이 거의 생기지 않는다.

### 1. 고객별 등장 구간

In [4]:
g = df.groupby("user_id")["event_time"]

span = DataFrame({
    "첫 등장"    : g.min(),
    "마지막 등장": g.max(),
    "이벤트 수"  : g.size(),
})

span["관측 기간(일)"] = (span["마지막 등장"] - span["첫 등장"]).dt.days
span.head()

,첫 등장,마지막 등장,이벤트 수,관측 기간(일)
user_id,,,,
1120748,2019-11-04 18:28:57,2019-11-04 18:28:57,1,0
1458813,2019-11-19 12:56:09,2019-11-21 09:02:40,29,1
4103071,2019-10-22 00:05:31,2019-10-22 00:24:19,8,0
5493470,2019-11-11 21:02:09,2019-11-11 21:10:16,10,0
6217356,2019-11-22 15:39:43,2019-11-22 15:39:43,1,0


### 2. 며칠에 걸쳐 활동했는가

- 같은 날 여러 번 들어온 것은 1일로 센다.
- 관측 기간이 길어도 활동 일수가 1이면, 첫날과 마지막 날 사이가 비어 있다는 뜻이다.

In [5]:
tmp = DataFrame({"user_id": df["user_id"],
                 "d"      : df["event_time"].dt.normalize()})

span["활동 일수"] = tmp.groupby("user_id")["d"].nunique()

del tmp
span[["이벤트 수", "관측 기간(일)", "활동 일수"]].describe()

,이벤트 수,관측 기간(일),활동 일수
count,712663.000000,712663.000000,712663.000000
mean,11.589151,3.484574,1.555514
std,48.990083,10.226662,1.898134
min,1.000000,0.000000,1.000000
25%,1.000000,0.000000,1.000000
50%,2.000000,0.000000,1.000000
75%,4.000000,0.000000,1.000000
max,8628.000000,60.000000,60.000000


### 3. 요약

In [6]:
n = len(span)

print(f"고객 수            : {n:,}")
print(f"활동 일수 2일 이상 : {(span['활동 일수'] > 1).sum():,}  ({(span['활동 일수'] > 1).mean():.1%})")
print(f"관측 기간 7일 이상 : {(span['관측 기간(일)'] >= 7).sum():,}  ({(span['관측 기간(일)'] >= 7).mean():.1%})")
print(f"관측 기간 최댓값   : {span['관측 기간(일)'].max()}일")

고객 수            : 712,663
활동 일수 2일 이상 : 150,794  (21.2%)
관측 기간 7일 이상 : 88,661  (12.4%)
관측 기간 최댓값   : 60일


### 4. 실제 분석 대상만 다시 보기

- 우리 분석 대상은 **`cart`를 낸 고객**이다. 전체 평균보다 이쪽 수치가 중요하다.
- 한 번 조회하고 떠난 사람이 다수라면 전체 평균이 낮게 나오는데, 그것은 우리 질문과 무관하다.

In [7]:
cart_users = df.loc[df["event_type"] == "cart", "user_id"].unique()
sub = span.loc[cart_users]

print(f"cart를 낸 고객     : {len(sub):,}")
print(f"활동 일수 2일 이상 : {(sub['활동 일수'] > 1).mean():.1%}")
print(f"관측 기간 7일 이상 : {(sub['관측 기간(일)'] >= 7).mean():.1%}")
print(f"관측 기간 최댓값   : {sub['관측 기간(일)'].max()}일")

cart를 낸 고객     : 210,593
활동 일수 2일 이상 : 37.1%
관측 기간 7일 이상 : 25.0%
관측 기간 최댓값   : 60일


### 5. 판정

| 결과 | 해석 | 다음 |
|---|---|---|
| 관측 기간 최댓값이 **50일 이상**이고, 여러 날에 걸친 고객이 상당수 | 지속되는 식별자다 | **통과.** Y 설계 그대로 |
| 거의 전부가 활동 일수 1일이고 관측 기간이 며칠 이내 | 방문마다 새로 발급되는 임시 ID다 | **Y 설계 재검토** |

데이터가 62일치이므로, 60일 가까이 걸친 `user_id`가 존재한다면
그것만으로도 사람 단위 식별자라는 근거가 된다.

> **→ 결정기록 2번**에 위 수치와 판정을 적는다.

## #03. 전제 B — `user_session`이 한 번의 방문인가
### 왜 확인하는가

설명변수 넷이 전부 "담기 전에 얼마나 봤나"를 세는 변수다.

`has_prior_view` · `prior_view_cnt` · `prior_uniq_product` · `same_product_view_cnt`

그런데 **어디까지를 "담기 전"으로 볼지**가 아직 정해지지 않았다. 후보가 둘이다.

- **세션 기준** — 같은 방문 안에서 t₀ 이전에 본 것만 센다
- **시간 창 기준** — t₀ 직전 N분 안에 본 것만 센다

세션 기준이 해석하기 훨씬 좋다. "이번에 들어와서 3개 보고 담았다"는 문장이 되기 때문이다.
그래서 세션이 정말 방문 단위이기만 하면 그쪽을 쓰는 것이 낫다.

**세션이 방문 단위가 아니면 무엇이 망가지는가**

- 세션 하나가 며칠에 걸쳐 있으면 → "같은 방문에서 본 조회 수"가 실제로는 며칠치 누적이 된다
- 반대로 몇 분 만에 세션이 갈리면 → 한 번의 방문이 여러 세션으로 쪼개져 조회 수가 실제보다 적게 잡힌다

둘 다 변수의 의미가 깨진다. 그러면 "탐색을 많이 한 사람이 더 산다"는 결론이
사실은 **세션 ID의 습성**을 본 것에 불과해진다.

### 1. 세션당 이벤트 수와 지속 시간

- `user_session` 결측 행(0.02%)은 `groupby`에서 자동으로 빠진다.

In [8]:
sess = df.groupby("user_session", observed=True)["event_time"].agg(["count", "min", "max"])
sess["지속(분)"] = (sess["max"] - sess["min"]).dt.total_seconds() / 60

print(f"세션 수 : {len(sess):,}")
sess[["count", "지속(분)"]].describe()

세션 수 : 1,806,092


,count,지속(분)
count,1.806092e+06,1.806092e+06
mean,4.572209e+00,8.541854e+01
std,1.310605e+01,1.715216e+03
min,1.000000e+00,0.000000e+00
25%,1.000000e+00,0.000000e+00
50%,1.000000e+00,0.000000e+00
75%,3.000000e+00,1.783333e+00
max,2.377000e+03,8.741905e+04


### 2. 한눈에 보기

- 이벤트가 1건뿐인 세션이 너무 많으면 방문이라 보기 어렵다.
- 반대로 며칠씩 이어지는 세션이 있으면 그것도 방문이 아니다.

In [9]:
print(f"이벤트 1건짜리 세션 : {(sess['count'] == 1).mean():.1%}")
print(f"지속 30분 미만      : {(sess['지속(분)'] <  30).mean():.1%}")
print(f"지속 24시간 초과    : {(sess['지속(분)'] > 1440).mean():.1%}")
print(f"최장 세션           : {sess['지속(분)'].max() / 60 / 24:.1f}일")

이벤트 1건짜리 세션 : 61.4%
지속 30분 미만      : 93.4%
지속 24시간 초과    : 0.6%
최장 세션           : 60.7일


### 3. 세션 경계의 시간 간격  ← **이것이 결정적이다**

한 사람의 기록을 시간순으로 늘어놓고, **세션 ID가 바뀌는 지점마다 직전 행동과의 시간 간격**을 잰다.

세션이 방문 단위라면 이 간격은 대체로 **30분 이상**이어야 한다.
자리를 떴다가 한참 뒤에 다시 들어와야 새 방문이기 때문이다.

이 간격이 **1분 미만인 경우가 대량**이면, 세션 ID는 시간과 무관하게 갈린다는 뜻이다.
즉 세션은 "방문"이 아니라 브라우저 탭이나 기기 구분 같은 다른 무언가다.

> 참고 — 01번 노트북에서 정황이 하나 나왔다.
> 4개 컬럼 기준 중복 정리에서 걸린 행들이 **"같은 사람이 같은 상품을 같은 초에 조회했는데
> 세션 ID가 둘로 갈린"** 케이스였다. 같은 초에 세션이 갈린다는 것은 시간 기반이 아니라는 신호다.

In [10]:
# 필요한 컬럼만 뽑아 정렬한다 (800만 행 전체를 정렬하면 메모리가 두 배로 든다)
d = (df[["user_id", "event_time", "user_session", "event_type"]]
       .sort_values(["user_id", "event_time"]))

d["gap_min"]   = d.groupby("user_id")["event_time"].diff().dt.total_seconds() / 60
d["prev_sess"] = d.groupby("user_id")["user_session"].shift()
d["new_sess"]  = d["user_session"] != d["prev_sess"]

# 같은 고객 안에서 세션이 바뀐 지점만,
# 그리고 앞뒤 세션 ID가 모두 존재하는 경우만 (결측이 끼면 '바뀐 것'으로 잘못 잡힌다)
mask = (d["new_sess"]
        & d["gap_min"].notna()
        & d["user_session"].notna()
        & d["prev_sess"].notna())

gap = d.loc[mask, "gap_min"]
print(f"세션 경계 건수 : {len(gap):,}")
gap.describe()

세션 경계 건수 : 1,202,648


count    1.202648e+06
mean     2.957061e+03
std      8.444965e+03
min      0.000000e+00
25%      5.000000e-01
50%      5.766667e+00
75%      1.255317e+03
max      8.690350e+04
Name: gap_min, dtype: float64

### 4. 간격의 분포

In [11]:
print(f"1분 미만   : {(gap <   1).mean():.1%}")
print(f"5분 미만   : {(gap <   5).mean():.1%}")
print(f"30분 미만  : {(gap <  30).mean():.1%}")
print(f"중앙값     : {gap.median():.2f}분")
print(f"최댓값     : {gap.max() / 60 / 24:.1f}일")

1분 미만   : 33.0%
5분 미만   : 49.0%
30분 미만  : 57.6%
중앙값     : 5.77분
최댓값     : 60.3일


### 5. 판정

| 결과 | 해석 | 조회 변수 구간 |
|---|---|---|
| 세션 경계 간격이 대체로 **30분 이상**, 1분 미만이 드물다 | 시간으로 끊긴 방문 단위다 | **`user_session` 기준** |
| **1분 미만이 상당 비율**이고 중앙값도 짧다 | 시간과 무관하게 갈린다. 방문이 아니다 | **t₀ 직전 시간 창 기준** |

> **→ 결정기록 3번 전반부**에 위 수치와 판정을 적는다.

## #04. 조회 변수 구간 확정  〈함께〉

`#03`의 판정에 따라 갈린다.

- **세션 = 방문 단위로 확인된 경우** → `user_session` 기준으로 확정. 아래 `#04-1`은 건너뛴다.
- **방문 단위가 아닌 경우** → 시간 창을 쓴다. 창 길이를 아래에서 정한다.

### 1. 시간 창 길이 정하기

창을 너무 짧게 잡으면 담기 전 탐색을 놓치고, 너무 길게 잡으면 관계없는 과거가 섞인다.

판단 재료로 **`cart` 직전 행동과의 시간 간격**을 본다.
"담기 직전의 탐색"이 대체로 몇 분 안에 일어나는지 보면 창 길이의 근거가 된다.

In [12]:
cart_gap = d.loc[(d["event_type"] == "cart") & d["gap_min"].notna(), "gap_min"]

print(f"cart 건수 : {len(cart_gap):,}")
print()
for m in [1, 5, 10, 30, 60, 180]:
    print(f"직전 행동과 {m:>3}분 이내 : {(cart_gap < m).mean():.1%}")
print()
print(cart_gap.describe())

cart 건수 : 2,431,971

직전 행동과   1분 이내 : 72.0%
직전 행동과   5분 이내 : 92.8%
직전 행동과  10분 이내 : 95.4%
직전 행동과  30분 이내 : 96.8%
직전 행동과  60분 이내 : 97.2%
직전 행동과 180분 이내 : 97.6%

count    2.431971e+06
mean     2.140577e+02
std      2.429144e+03
min      0.000000e+00
25%      1.166667e-01
50%      3.500000e-01
75%      1.150000e+00
max      8.576932e+04
Name: gap_min, dtype: float64


In [13]:
# ---------------------------------------------------------
# 창 길이별 has_prior_view = 0 비율
#
#  각 cart에 대해 '직전 view'가 언제였는지 찾고,
#  그 간격이 창 안에 들어오는지를 창 길이별로 센다.
#  창이 길수록 0이 줄어드는데, 0이 너무 줄면 이 변수가 변별력을 잃는다.
# ---------------------------------------------------------
from pandas import merge_asof

carts = (df.loc[df["event_type"] == "cart", ["user_id", "event_time"]]
           .sort_values("event_time")
           .reset_index(drop=True))

views = (df.loc[df["event_type"] == "view", ["user_id", "event_time"]]
           .sort_values("event_time")
           .rename(columns={"event_time": "view_time"}))

# 같은 고객의 기록 중, 각 cart보다 앞선 view 하나를 찾는다
m = merge_asof(carts, views,
               left_on="event_time", right_on="view_time",
               by="user_id", direction="backward")

m["gap"] = (m["event_time"] - m["view_time"]).dt.total_seconds() / 60

print(f"cart 건수 : {len(m):,}\n")
print(" 창    has_prior_view=0    =1")
for n in [5, 10, 30, 60, 180]:
    zero = 1 - (m["gap"] <= n).mean()
    print(f"{n:>3}분      {zero:6.1%}        {1-zero:6.1%}")

cart 건수 : 2,490,795

 창    has_prior_view=0    =1
  5분       48.6%         51.4%
 10분       38.8%         61.2%
 30분       30.4%         69.6%
 60분       28.3%         71.7%
180분       26.4%         73.6%


### 2. 창 길이 판단

- 위 표에서 **비율이 완만해지는 지점**이 창 길이의 후보다.
  예를 들어 30분에서 60분으로 늘려도 몇 %밖에 안 늘어난다면, 30분으로 충분하다는 뜻이다.
- 창을 정한 뒤에는 **결과를 보고 바꾸지 않는다.** 여기서 확정한다.

### 3. 결정

```
☐ user_session 기준
☐ t₀ 직전 시간 창 기준  →  창 길이 : ______ 분
```

**→ 결정기록 3번 후반부**에 결정과 근거를 적는다.

> 이 결정으로 아래 네 변수의 정의가 확정된다.
>
> | 변수 | 정의 |
> |---|---|
> | `has_prior_view` | 구간 안에 t₀ 이전 조회가 **있었는지** (0/1) |
> | `prior_view_cnt` | 구간 안 t₀ 이전 조회 **건수** |
> | `prior_uniq_product` | 구간 안 t₀ 이전 조회한 **상품 종류 수** |
> | `same_product_view_cnt` | 구간 안에서 **담은 그 상품을** t₀ 이전에 조회한 횟수 |

## #05. 정리

### 1. 결정기록에 옮길 값

| 항목 | 채울 것 |
|---|---|
| 2번 | 2일 이상 활동 비율 · 관측 기간 최댓값 · cart 고객 기준 수치 · 판정 |
| 3번 전반부 | 세션당 이벤트 수 · 지속 시간 · **경계 간격**(1분 미만 / 30분 미만 비율, 중앙값, 최댓값) · 판정 |
| 3번 후반부 | 조회 변수 구간 결정과 근거 (시간 창이면 길이까지) |

### 2. 다음 단계

**03번 노트북 — 분석 대상 테이블 생성.**
여기서 확정한 구간 정의로 1행 = 1고객 테이블과 목표변수를 만든다.

### 3. 메모리 정리

In [14]:
del d, gap, sess, span, sub
import gc
gc.collect()

60